# 03a - Encode Frozen VLM Features Offline On RTX6000 - batch04

Run this notebook on Kaggle Nemotron RTX6000 with internet disabled. It reads an offline Eagle2-2B cache produced by `03a0_cache_eagle2_2b_for_offline.ipynb`, then encodes frozen VLM features from video frames + task text.

Required inputs:
- Output of `03a0_cache_eagle2_2b_for_offline.ipynb`: `eagle2_offline_cache`.
- 4 Notebook 02 batch outputs: `gr00t_prepared_official_H16_batch01..04`.
- 20 Notebook 01 download outputs containing partial video chunks.

Main outputs:
- `gr00t_vlm_encoded_H16_20subsets_rtx6000_offline/vl_features_train.npy`
- `gr00t_vlm_encoded_H16_20subsets_rtx6000_offline/vl_feature_index_train.parquet`
- `gr00t_vlm_encoded_H16_20subsets_rtx6000_offline/vl_features_test.npy`
- `gr00t_vlm_encoded_H16_20subsets_rtx6000_offline/encode_summary.json`

Default scale for RTX6000:
- `MAX_ENCODE_TRAIN_SAMPLES=654143`
- `MAX_ENCODE_TEST_SAMPLES=163425`
- `ENCODE_BATCH_SIZE=8`

Override these with environment variables if Kaggle time/disk becomes tight.

This batch notebook encodes only `batch04`. Combine all 4 encoded outputs before Notebook 03b training.

In [1]:
# Offline dependency install from the cache notebook output. No internet is required.
from pathlib import Path
import os, subprocess, sys

os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"

def find_eagle2_cache_root():
    explicit = os.getenv("EAGLE2_CACHE_ROOT")
    if explicit and Path(explicit).exists():
        return Path(explicit)
    candidates = []
    for pattern in [
        "/kaggle/input/notebooks/*/*/eagle2_offline_cache",
        "/kaggle/input/*/eagle2_offline_cache",
        "/kaggle/working/eagle2_offline_cache",
    ]:
        candidates.extend(Path("/").glob(pattern.lstrip("/")))
    candidates = [p for p in candidates if (p / "hf_models" / "nvidia_Eagle2-2B").exists()]
    if not candidates:
        raise FileNotFoundError("Could not find eagle2_offline_cache. Attach the output of 03a0_cache_eagle2_2b_for_offline.ipynb.")
    return sorted(candidates, key=lambda p: len(str(p)))[0]

EAGLE2_CACHE_ROOT = find_eagle2_cache_root()
EAGLE2_MODEL_PATH = EAGLE2_CACHE_ROOT / "hf_models" / "nvidia_Eagle2-2B"
EAGLE2_WHEELHOUSE = EAGLE2_CACHE_ROOT / "wheelhouse"
print("EAGLE2_CACHE_ROOT:", EAGLE2_CACHE_ROOT)
print("EAGLE2_MODEL_PATH:", EAGLE2_MODEL_PATH)
print("EAGLE2_WHEELHOUSE:", EAGLE2_WHEELHOUSE)

if not EAGLE2_WHEELHOUSE.exists():
    raise FileNotFoundError(f"Missing wheelhouse: {EAGLE2_WHEELHOUSE}")

# Install only user-space packages from the wheelhouse.
# Do not let pip resolve dependencies here, otherwise it may upgrade torch/torchvision/cuda
# on Kaggle RTX6000 and break torchvision custom ops.
offline_packages = [
    "transformers==4.51.0",
    "huggingface_hub>=0.30.0,<1.0",
    "accelerate>=1.7.0",
    "tokenizers>=0.21,<0.22",
    "safetensors>=0.4.3",
    "decord>=0.6.0",
    "av",
    "opencv-python-headless",
]
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "--no-index", "--no-deps", "--find-links", str(EAGLE2_WHEELHOUSE),
    *offline_packages,
])

EAGLE2_CACHE_ROOT: /kaggle/input/notebooks/kimthanh211005/gr00t-eagle2-2b-offline-cache/eagle2_offline_cache
EAGLE2_MODEL_PATH: /kaggle/input/notebooks/kimthanh211005/gr00t-eagle2-2b-offline-cache/eagle2_offline_cache/hf_models/nvidia_Eagle2-2B
EAGLE2_WHEELHOUSE: /kaggle/input/notebooks/kimthanh211005/gr00t-eagle2-2b-offline-cache/eagle2_offline_cache/wheelhouse


0

In [2]:
from pathlib import Path
import json, random, time, os, math, hashlib
import numpy as np, pandas as pd
from PIL import Image
from tqdm.auto import tqdm
import torch
import torch.nn.functional as F

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print("gpu:", GPU_NAME)
REQUIRE_RTX6000_GPU = os.getenv("REQUIRE_RTX6000_GPU", "1") == "1"
if REQUIRE_RTX6000_GPU and torch.cuda.is_available() and "6000" not in GPU_NAME:
    raise RuntimeError(
        f"This offline encode notebook must run on RTX6000/NvidiaRtxPro6000, but Kaggle assigned: {GPU_NAME}. "
        "Re-run with accelerator NvidiaRtxPro6000 in the Nemotron competition runtime."
    )

SMOKE_TEST = False
ACTION_HORIZON = 16
STATE_HORIZON = 1
ENCODE_BATCH_NAME = os.getenv("ENCODE_BATCH_NAME", "batch04_full")
MAX_ENCODE_TRAIN_SAMPLES = 512 if SMOKE_TEST else int(os.getenv("MAX_ENCODE_TRAIN_SAMPLES", "654143"))
MAX_ENCODE_TEST_SAMPLES = 256 if SMOKE_TEST else int(os.getenv("MAX_ENCODE_TEST_SAMPLES", "163425"))
ENCODE_BATCH_SIZE = int(os.getenv("ENCODE_BATCH_SIZE", "8"))
USE_AMP = True
PRIMARY_ENCODER = os.getenv("PRIMARY_ENCODER", str(EAGLE2_MODEL_PATH))
FALLBACK_ENCODER = "google/siglip-base-patch16-224"
FORCE_FALLBACK = False
HF_TOKEN_SECRET_NAME = os.getenv("HF_TOKEN_SECRET_NAME", "HF_TOKEN")
HF_TOKEN_REQUIRED_FOR_PRIMARY = os.getenv("HF_TOKEN_REQUIRED_FOR_PRIMARY", "0") == "1"
HF_TOKEN = None
REQUIRE_PRIMARY_ENCODER = os.getenv("REQUIRE_PRIMARY_ENCODER", "1") == "1"
FEATURE_STORAGE_DTYPE = np.float16
ENCODE_CACHE_VERSION = "vl_encode_multi_root_video_memmap_v1"
OUTPUT_ROOT = Path(f"/kaggle/working/gr00t_vlm_encoded_H{ACTION_HORIZON}_{ENCODE_BATCH_NAME}_rtx6000_offline")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


device: cuda
gpu: NVIDIA RTX PRO 6000 Blackwell Server Edition


In [3]:
# Offline mode: the Eagle2 snapshot is loaded from the attached cache input.
HF_TOKEN = None
print("Offline HF mode enabled; model path:", PRIMARY_ENCODER)

Offline HF mode enabled; model path: /kaggle/input/notebooks/kimthanh211005/gr00t-eagle2-2b-offline-cache/eagle2_offline_cache/hf_models/nvidia_Eagle2-2B


In [4]:
REQUIRED_PREPARED_FILES = [
    "states_train.npy", "actions_train_chunk.npy", "action_mask_train.npy",
    "video_frame_manifest_train.parquet", "video_frame_manifest_test.parquet",
    "normalization_stats.json", "prepare_report.json",
]

def _has_required(root):
    return root.is_dir() and all((root / name).exists() for name in REQUIRED_PREPARED_FILES)

def find_prepared_roots(action_horizon):
    exact = f"gr00t_prepared_official_H{action_horizon}"
    prefix = f"gr00t_prepared_official_H{action_horizon}_batch"
    candidates = []

    # Fast path for Kaggle notebook inputs: /kaggle/input/notebooks/<owner>/<slug>/<output_root>
    nb_base = Path("/kaggle/input/notebooks")
    if nb_base.exists():
        for source_dir in nb_base.glob("*/*"):
            if not source_dir.is_dir():
                continue
            for c in source_dir.iterdir():
                if c.name == exact or c.name.startswith(prefix):
                    if _has_required(c):
                        candidates.append(c)

    # Local/working fallback only. Avoid recursive scan over all video inputs unless necessary.
    for base in [Path("/kaggle/working")]:
        if not base.exists():
            continue
        for c in base.glob(f"gr00t_prepared_official_H{action_horizon}*"):
            if c.name == exact or c.name.startswith(prefix):
                if _has_required(c):
                    candidates.append(c)

    # Last-resort compatibility fallback.
    if not candidates:
        for base in [Path("/kaggle/input"), Path("/kaggle/working")]:
            if not base.exists():
                continue
            for c in base.rglob(f"gr00t_prepared_official_H{action_horizon}*"):
                if c.name == exact or c.name.startswith(prefix):
                    if _has_required(c):
                        candidates.append(c)

    dedup = {}
    for c in candidates:
        dedup[str(c.resolve())] = c
    roots = list(dedup.values())
    batch_roots = [r for r in roots if r.name.startswith(prefix)]
    if batch_roots:
        roots = batch_roots
    roots = sorted(roots, key=lambda x: str(x))
    if not roots:
        raise FileNotFoundError(f"Could not find prepared root H{action_horizon}. Mount Notebook 02 batch outputs or a merged prepared root.")
    return roots

def load_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def combine_normalization_stats(stats_list):
    def combine(mean_key, std_key, n_key):
        total_n = int(sum(int(s.get(n_key, 0)) for s in stats_list))
        if total_n <= 0:
            raise ValueError(f"Invalid normalization count for {n_key}")
        sum_x = None; sum_x2 = None
        for s in stats_list:
            n = int(s.get(n_key, 0))
            if n <= 0:
                continue
            mean = np.asarray(s[mean_key], dtype=np.float64)
            std = np.asarray(s[std_key], dtype=np.float64)
            sx = mean * n
            sx2 = (std ** 2 + mean ** 2) * n
            sum_x = sx if sum_x is None else sum_x + sx
            sum_x2 = sx2 if sum_x2 is None else sum_x2 + sx2
        mean = sum_x / total_n
        var = np.maximum(sum_x2 / total_n - mean ** 2, 1e-12)
        std = np.sqrt(var)
        std = np.where(std < 1e-6, 1.0, std)
        return mean.astype(np.float32).tolist(), std.astype(np.float32).tolist(), total_n
    sm, ss, sn = combine("state_mean", "state_std", "state_n")
    am, ast, an = combine("action_mean", "action_std", "action_n")
    return {"state_mean": sm, "state_std": ss, "action_mean": am, "action_std": ast,
            "state_n": sn, "action_n": an, "space": "combined_raw_train_split"}

PREPARED_ROOTS = find_prepared_roots(ACTION_HORIZON)
PREPARED_ROOT_BY_ID = {i: r for i, r in enumerate(PREPARED_ROOTS)}
prepare_reports = [load_json(r / "prepare_report.json") for r in PREPARED_ROOTS]
stats_list = [load_json(r / "normalization_stats.json") for r in PREPARED_ROOTS]
stats = combine_normalization_stats(stats_list)
prepare_report = {
    "status": "combined",
    "state_horizon": STATE_HORIZON,
    "action_horizon": ACTION_HORIZON,
    "num_prepared_roots": len(PREPARED_ROOTS),
    "prepared_roots": [str(r) for r in PREPARED_ROOTS],
    "train_samples": int(sum(int(x.get("train_samples", 0)) for x in prepare_reports)),
    "test_samples": int(sum(int(x.get("test_samples", 0)) for x in prepare_reports)),
}
root_digest = hashlib.sha1("|".join(str(r) for r in PREPARED_ROOTS).encode("utf-8")).hexdigest()[:10]
print("PREPARED_ROOTS:")
for i, r in enumerate(PREPARED_ROOTS):
    rep = prepare_reports[i]
    print(f"  [{i}] {r} train={rep.get('train_samples')} test={rep.get('test_samples')}")
print(json.dumps({k: prepare_report[k] for k in ["num_prepared_roots", "train_samples", "test_samples"]}, indent=2))


PREPARED_ROOTS:
  [0] /kaggle/input/notebooks/kimthanh211005/gr00t-prepare-h16-batch04-5-subsets/gr00t_prepared_official_H16_batch04 train=654143 test=163425
{
  "num_prepared_roots": 1,
  "train_samples": 654143,
  "test_samples": 163425
}


In [5]:
def decode_frame_pyav(video_path, frame_index):
    import av
    container = av.open(video_path)
    try:
        stream = container.streams.video[0]
        last = None
        for i, frame in enumerate(container.decode(stream)):
            last = frame
            if i >= frame_index:
                return frame.to_image().convert("RGB")
        if last is None: raise RuntimeError("empty video")
        return last.to_image().convert("RGB")
    finally:
        container.close()

def decode_frame_cv2(video_path, frame_index):
    import cv2
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened(): raise RuntimeError("cv2 cannot open video")
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_index))
    ok, frame = cap.read()
    if not ok:
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        cap.set(cv2.CAP_PROP_POS_FRAMES, max(0, total - 1)); ok, frame = cap.read()
    cap.release()
    if not ok: raise RuntimeError("cv2 cannot decode frame")
    return Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

def decode_frame(video_path, frame_index):
    # Uu tien PyAV vi doc video on dinh; neu loi thi fallback sang OpenCV.
    try: return decode_frame_pyav(video_path, int(frame_index))
    except Exception: return decode_frame_cv2(video_path, int(frame_index))

In [6]:
# Compatibility shim for NVIDIA Eagle2 custom processor.
# Kaggle/Transformers combinations can miss type aliases/docstring constants expected by Eagle2 remote code.
from typing import Any
import inspect
import sys
import types
import transformers.image_utils as _hf_image_utils
if not hasattr(_hf_image_utils, "VideoInput"):
    _hf_image_utils.VideoInput = Any
    print("Patched transformers.image_utils.VideoInput for Eagle2 processor compatibility")
try:
    import transformers.image_processing_utils_fast as _hf_fast_original
    _hf_fast_utils = types.ModuleType("transformers.image_processing_utils_fast")
    for _name in dir(_hf_fast_original):
        try:
            setattr(_hf_fast_utils, _name, getattr(_hf_fast_original, _name))
        except Exception:
            pass
    _hf_fast_utils.__dict__.setdefault("BASE_IMAGE_PROCESSOR_FAST_DOCSTRING", "")
    _hf_fast_utils.__dict__.setdefault("BASE_IMAGE_PROCESSOR_FAST_DOCSTRING_PREPROCESS", "")
    _hf_fast_utils.__dict__.setdefault("__file__", getattr(_hf_fast_original, "__file__", None))
    _hf_fast_utils.__dict__.setdefault("__spec__", getattr(_hf_fast_original, "__spec__", None))
    sys.modules["transformers.image_processing_utils_fast"] = _hf_fast_utils
    print("Patched transformers.image_processing_utils_fast for Eagle2 remote code")
except Exception as exc:
    print(f"Fast image processor shim skipped: {type(exc).__name__}: {exc}")

from transformers import AutoConfig, AutoModel, AutoProcessor
from transformers.modeling_utils import PreTrainedModel

def _disable_flash_attn2_check(cls, config, *args, **kwargs):
    for obj in [config, getattr(config, "vision_config", None), getattr(config, "text_config", None), getattr(config, "llm_config", None)]:
        if obj is None:
            continue
        for attr in ["_attn_implementation", "attn_implementation"]:
            try:
                setattr(obj, attr, "eager")
            except Exception:
                pass
    return config

PreTrainedModel._check_and_enable_flash_attn_2 = classmethod(_disable_flash_attn2_check)
print("Patched Transformers FlashAttention2 check to use eager attention when flash_attn is unavailable")

if torch.distributed.is_available() and not torch.distributed.is_initialized():
    torch.distributed.get_rank = lambda group=None: 0
    print("Patched torch.distributed.get_rank for single-GPU Eagle2 inference")

class FrozenVLEncoder:
    def __init__(self, primary_id, fallback_id, force_fallback=False):
        self.primary_id = primary_id
        self.fallback_id = fallback_id
        self.force_fallback = force_fallback
        self.encoder_name = None
        self.fallback_used = False
        self.processor = None
        self.model = None
        self.feature_dim = None
        self.num_tokens = 2
        self.primary_error = None
        self.load()

    def _auth_kwargs(self):
        return {"token": HF_TOKEN} if HF_TOKEN else {}

    def _force_eager_attention(self, config):
        for obj in [config, getattr(config, "vision_config", None), getattr(config, "text_config", None), getattr(config, "llm_config", None)]:
            if obj is None:
                continue
            for attr in ["_attn_implementation", "attn_implementation"]:
                try:
                    setattr(obj, attr, "eager")
                except Exception:
                    pass
        return config

    def try_load(self, model_id):
        kwargs = {"trust_remote_code": True, "local_files_only": True, **self._auth_kwargs()}
        proc = AutoProcessor.from_pretrained(model_id, use_fast=False, **kwargs)
        if hasattr(proc, "tokenizer"):
            proc.tokenizer.padding_side = "left"
        config = AutoConfig.from_pretrained(model_id, **kwargs)
        config = self._force_eager_attention(config)
        model = AutoModel.from_pretrained(
            model_id,
            config=config,
            trust_remote_code=True,
            low_cpu_mem_usage=True,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            attn_implementation="eager",
            use_flash_attention_2=False,
            **self._auth_kwargs(),
        )
        model.eval().to(device)
        for p in model.parameters():
            p.requires_grad_(False)
        return proc, model

    def load(self):
        if not self.force_fallback:
            try:
                self.processor, self.model = self.try_load(self.primary_id)
                self.encoder_name = self.primary_id
                print("Loaded primary encoder:", self.encoder_name)
                return
            except Exception as exc:
                self.primary_error = repr(exc)
                print("ERROR primary encoder failed:", self.primary_error)
                if REQUIRE_PRIMARY_ENCODER:
                    raise RuntimeError(
                        f"Primary encoder is required but failed to load: {self.primary_id}. "
                        f"Check HF_TOKEN and model access. Original error: {self.primary_error}"
                    )
        self.processor, self.model = self.try_load(self.fallback_id)
        self.encoder_name = self.fallback_id
        self.fallback_used = True
        print("Loaded fallback encoder:", self.encoder_name)

    def _filter_forward_inputs(self, inputs):
        try:
            allowed = set(inspect.signature(self.model.forward).parameters)
            return {k: v for k, v in inputs.items() if k in allowed}
        except Exception:
            return {k: v for k, v in inputs.items() if k != "image_sizes"}

    def _build_vlm_inputs(self, images, texts):
        if hasattr(self.processor, "apply_chat_template") and hasattr(self.processor, "process_vision_info"):
            conversations = [
                [{
                    "role": "user",
                    "content": [
                        {"type": "image", "image": image},
                        {"type": "text", "text": text},
                    ],
                }]
                for image, text in zip(images, texts)
            ]
            text_list = [
                self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
                for messages in conversations
            ]
            vision = self.processor.process_vision_info(conversations)
            if isinstance(vision, tuple) and len(vision) == 3:
                image_inputs, video_inputs, video_kwargs = vision
                return self.processor(
                    text=text_list,
                    images=image_inputs,
                    videos=video_inputs,
                    return_tensors="pt",
                    padding=True,
                    videos_kwargs=video_kwargs,
                )
            image_inputs, video_inputs = vision
            return self.processor(
                text=text_list,
                images=image_inputs,
                videos=video_inputs,
                return_tensors="pt",
                padding=True,
            )
        return self.processor(text=texts, images=images, padding=True, return_tensors="pt")

    @torch.no_grad()
    def encode_batch(self, images, texts):
        # Encoder is fully frozen; Notebook 03b trains only the DiT/action head from scratch.
        inputs = self._build_vlm_inputs(images, texts)
        inputs = {k: v.to(device) if hasattr(v, "to") else v for k, v in inputs.items()}
        if "pixel_values" in inputs and "image_flags" not in inputs:
            inputs["image_flags"] = torch.ones(
                (int(inputs["pixel_values"].shape[0]), 1),
                dtype=torch.long,
                device=inputs["pixel_values"].device,
            )
        with torch.cuda.amp.autocast(enabled=USE_AMP and torch.cuda.is_available()):
            if hasattr(self.model, "get_image_features") and hasattr(self.model, "get_text_features") and "pixel_values" in inputs:
                image_feat = self.model.get_image_features(pixel_values=inputs["pixel_values"])
                text_keys = {k: v for k, v in inputs.items() if k in ["input_ids", "attention_mask"]}
                text_feat = self.model.get_text_features(**text_keys) if text_keys else torch.zeros_like(image_feat)
                feats = torch.stack([F.normalize(image_feat.float(), dim=-1), F.normalize(text_feat.float(), dim=-1)], 1)
            else:
                forward_inputs = self._filter_forward_inputs(inputs)
                out = self.model(**forward_inputs, output_hidden_states=True, return_dict=True)
                hidden = getattr(out, "last_hidden_state", None)
                if hidden is None:
                    hidden = out.hidden_states[-1]
                pooled = hidden.float().mean(1)
                feats = torch.stack([pooled, pooled], 1)
        self.feature_dim = int(feats.shape[-1])
        self.num_tokens = int(feats.shape[1])
        return feats.cpu().numpy().astype(np.float16)


2026-06-20 02:36:55.609973: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781923015.781505      65 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781923015.829895      65 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781923016.263768      65 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781923016.263782      65 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781923016.263783      65 computation_placer.cc:177] computation placer alr

Patched transformers.image_processing_utils_fast for Eagle2 remote code
Patched Transformers FlashAttention2 check to use eager attention when flash_attn is unavailable
Patched torch.distributed.get_rank for single-GPU Eagle2 inference


In [7]:
def _balanced_sample(df, max_samples, group_col="subset"):
    if max_samples is None or len(df) <= int(max_samples):
        return df.reset_index(drop=True)
    max_samples = int(max_samples)
    groups = [(k, g.copy()) for k, g in df.groupby(group_col, sort=True)]
    quota = max(1, max_samples // max(len(groups), 1))
    selected, leftovers = [], []
    for _, g in groups:
        if len(g) <= quota:
            selected.append(g)
        else:
            selected.append(g.sample(n=quota, random_state=SEED))
            leftovers.append(g.drop(selected[-1].index))
    out = pd.concat(selected, ignore_index=False) if selected else df.iloc[:0]
    remaining = max_samples - len(out)
    if remaining > 0 and leftovers:
        rest = pd.concat(leftovers, ignore_index=False)
        if len(rest) > remaining:
            rest = rest.sample(n=remaining, random_state=SEED + 1)
        out = pd.concat([out, rest], ignore_index=False)
    return out.sort_values(["root_id", "subset", "episode_index", "frame_index", "sample_id"]).reset_index(drop=True)

def select_manifest(split, max_samples):
    frames = []
    for root_id, root in PREPARED_ROOT_BY_ID.items():
        df = pd.read_parquet(root / f"video_frame_manifest_{split}.parquet")
        df = df[(df.has_video == True) & (df.video_path.astype(str).str.len() > 0)].copy()
        if df.empty:
            continue
        df["root_id"] = int(root_id)
        df["root_name"] = root.name
        df["sample_id"] = df["sample_id"].astype(np.int64)
        frames.append(df)
    if not frames:
        raise RuntimeError(f"No video-backed samples for {split}. Notebook 03 needs Notebook 01 video outputs mounted, not only Notebook 02 arrays.")
    manifest = pd.concat(frames, ignore_index=True)
    manifest = manifest.sort_values(["root_id", "subset", "episode_index", "frame_index", "sample_id"]).reset_index(drop=True)
    manifest = _balanced_sample(manifest, max_samples, group_col="subset")
    return manifest

VIDEO_INDEX = None
VIDEO_PATH_CACHE = {}

def build_video_index():
    global VIDEO_INDEX
    if VIDEO_INDEX is not None:
        return VIDEO_INDEX
    VIDEO_INDEX = {}
    base = Path("/kaggle/input")
    if base.exists():
        for ext in ("*.mp4", "*.mkv", "*.avi", "*.mov"):
            for p in base.rglob(ext):
                VIDEO_INDEX.setdefault(p.name, []).append(p)
    print("indexed video files:", sum(len(v) for v in VIDEO_INDEX.values()))
    return VIDEO_INDEX

def resolve_video_path(video_path, subset=None):
    raw = str(video_path)
    if raw in VIDEO_PATH_CACHE:
        return VIDEO_PATH_CACHE[raw]
    p = Path(raw)
    if p.exists():
        VIDEO_PATH_CACHE[raw] = str(p)
        return str(p)
    idx = build_video_index()
    candidates = idx.get(p.name, [])
    subset = str(subset or "")
    if subset:
        subset_candidates = [c for c in candidates if subset in str(c)]
        if subset_candidates:
            VIDEO_PATH_CACHE[raw] = str(subset_candidates[0])
            return str(subset_candidates[0])
    if candidates:
        VIDEO_PATH_CACHE[raw] = str(candidates[0])
        return str(candidates[0])
    raise FileNotFoundError(f"Video not mounted or not found: {raw}")

def _cache_valid(split, feat_path, idx_path, report_path, expected_rows):
    if not (feat_path.exists() and idx_path.exists() and report_path.exists()):
        return False
    try:
        rep = json.loads(report_path.read_text(encoding="utf-8"))
        idx = pd.read_parquet(idx_path, columns=["root_id", "sample_id", "feature_index"])
        feat = np.load(feat_path, mmap_mode="r")
        return (rep.get("cache_version") == ENCODE_CACHE_VERSION and
                rep.get("root_digest") == root_digest and
                int(rep.get("encoded", -1)) == int(len(idx)) == int(feat.shape[0]) and
                set(["root_id", "sample_id", "feature_index"]).issubset(idx.columns))
    except Exception:
        return False

def encode_split(split, max_samples, encoder, batch_size=ENCODE_BATCH_SIZE):
    feat_path = OUTPUT_ROOT / f"vl_features_{split}.npy"
    idx_path = OUTPUT_ROOT / f"vl_feature_index_{split}.parquet"
    report_path = OUTPUT_ROOT / f"video_decode_report_{split}.json"
    manifest = select_manifest(split, max_samples)
    if _cache_valid(split, feat_path, idx_path, report_path, len(manifest)):
        return feat_path, idx_path, json.loads(report_path.read_text(encoding="utf-8"))

    rows, failed, feature_mm, encoded = [], [], None, 0
    expected = len(manifest)
    for start in tqdm(range(0, expected, batch_size), desc=f"encode {split}"):
        batch = manifest.iloc[start:start+batch_size]
        images, texts, ok = [], [], []
        for _, r in batch.iterrows():
            try:
                real_video_path = resolve_video_path(r.video_path, r.subset)
                images.append(decode_frame(real_video_path, int(r.frame_index)))
                txt = str(r.task_text) if str(r.task_text) and str(r.task_text) != "nan" else f"perform {r.subset}"
                texts.append(txt)
                ok.append((r, real_video_path))
            except Exception as exc:
                failed.append({"root_id": int(r.root_id), "sample_id": int(r.sample_id), "subset": str(r.subset), "error": str(exc)})
        if not images:
            continue
        feats = encoder.encode_batch(images, texts).astype(FEATURE_STORAGE_DTYPE, copy=False)
        if feature_mm is None:
            feature_mm = np.lib.format.open_memmap(feat_path, mode="w+", dtype=FEATURE_STORAGE_DTYPE,
                                                   shape=(expected, feats.shape[1], feats.shape[2]))
        n = feats.shape[0]
        feature_mm[encoded:encoded+n] = feats
        for j, (r, real_video_path) in enumerate(ok):
            rows.append({"feature_index": int(encoded + j), "root_id": int(r.root_id), "root_name": str(r.root_name),
                         "sample_id": int(r.sample_id), "subset": str(r.subset),
                         "episode_index": int(r.episode_index), "frame_index": int(r.frame_index),
                         "task_text": str(r.task_text), "video_path_resolved": str(real_video_path)})
        encoded += n
    if feature_mm is None or encoded == 0:
        raise RuntimeError(f"No frames encoded for {split}. Check that Notebook 01 video outputs are attached as inputs.")
    feature_mm.flush(); del feature_mm
    if encoded < expected:
        arr = np.load(feat_path, mmap_mode="r")[:encoded].copy()
        np.save(feat_path, arr.astype(FEATURE_STORAGE_DTYPE, copy=False))
    pd.DataFrame(rows).to_parquet(idx_path, index=False)
    report = {"split": split, "cache_version": ENCODE_CACHE_VERSION, "root_digest": root_digest,
              "prepared_roots": [str(r) for r in PREPARED_ROOTS], "requested": int(expected), "encoded": int(encoded),
              "failed": int(len(failed)), "feature_shape": list(np.load(feat_path, mmap_mode="r").shape),
              "feature_dtype": str(np.load(feat_path, mmap_mode="r").dtype),
              "encoder_name": encoder.encoder_name, "fallback_used": encoder.fallback_used,
              "primary_encoder": PRIMARY_ENCODER, "fallback_encoder": FALLBACK_ENCODER,
              "primary_error": getattr(encoder, "primary_error", None), "failed_examples": failed[:20],
              "max_samples": max_samples, "balanced_sampling_by_subset": True}
    report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
    return feat_path, idx_path, report

encoder = FrozenVLEncoder(PRIMARY_ENCODER, FALLBACK_ENCODER, FORCE_FALLBACK)
train_feat, train_idx, train_enc_report = encode_split("train", MAX_ENCODE_TRAIN_SAMPLES, encoder)
test_feat, test_idx, test_enc_report = encode_split("test", MAX_ENCODE_TEST_SAMPLES, encoder)
print(train_enc_report); print(test_enc_report)


Sliding Window Attention is enabled but not implemented for `eager`; unexpected results may be encountered.


Loaded primary encoder: /kaggle/input/notebooks/kimthanh211005/gr00t-eagle2-2b-offline-cache/eagle2_offline_cache/hf_models/nvidia_Eagle2-2B


encode train:   0%|          | 0/81768 [00:00<?, ?it/s]

/tmp/ipykernel_65/1658613587.py:174: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP and torch.cuda.is_available()):


dynamic ViT batch size: 8, images per sample: 1.0, dynamic token length: 302
dynamic ViT batch size: 8, images per sample: 1.0, dynamic token length: 302
dynamic ViT batch size: 8, images per sample: 1.0, dynamic token length: 302
dynamic ViT batch size: 8, images per sample: 1.0, dynamic token length: 302
dynamic ViT batch size: 8, images per sample: 1.0, dynamic token length: 302
dynamic ViT batch size: 8, images per sample: 1.0, dynamic token length: 302
dynamic ViT batch size: 8, images per sample: 1.0, dynamic token length: 301
dynamic ViT batch size: 8, images per sample: 1.0, dynamic token length: 301
dynamic ViT batch size: 8, images per sample: 1.0, dynamic token length: 301
dynamic ViT batch size: 8, images per sample: 1.0, dynamic token length: 301
dynamic ViT batch size: 8, images per sample: 1.0, dynamic token length: 301
dynamic ViT batch size: 8, images per sample: 1.0, dynamic token length: 302
dynamic ViT batch size: 8, images per sample: 1.0, dynamic token length: 302

encode test:   0%|          | 0/20429 [00:00<?, ?it/s]

/tmp/ipykernel_65/1658613587.py:174: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP and torch.cuda.is_available()):


dynamic ViT batch size: 8, images per sample: 1.0, dynamic token length: 301
dynamic ViT batch size: 8, images per sample: 1.0, dynamic token length: 301
dynamic ViT batch size: 8, images per sample: 1.0, dynamic token length: 301
dynamic ViT batch size: 8, images per sample: 1.0, dynamic token length: 301
dynamic ViT batch size: 8, images per sample: 1.0, dynamic token length: 301
dynamic ViT batch size: 8, images per sample: 1.0, dynamic token length: 301
dynamic ViT batch size: 8, images per sample: 1.0, dynamic token length: 301
dynamic ViT batch size: 8, images per sample: 1.0, dynamic token length: 301
dynamic ViT batch size: 8, images per sample: 1.0, dynamic token length: 301
dynamic ViT batch size: 8, images per sample: 1.0, dynamic token length: 301
dynamic ViT batch size: 8, images per sample: 1.0, dynamic token length: 301
dynamic ViT batch size: 8, images per sample: 1.0, dynamic token length: 301
dynamic ViT batch size: 8, images per sample: 1.0, dynamic token length: 301

In [8]:
summary = {
    "status": "completed",
    "output_root": str(OUTPUT_ROOT),
    "action_horizon": ACTION_HORIZON,
    "state_horizon": STATE_HORIZON,
    "num_prepared_roots": len(PREPARED_ROOTS),
    "prepared_roots": [str(r) for r in PREPARED_ROOTS],
    "root_digest": root_digest,
    "encoder_name": encoder.encoder_name,
    "fallback_used": bool(encoder.fallback_used),
    "primary_error": getattr(encoder, "primary_error", None),
    "train": train_enc_report,
    "test": test_enc_report,
    "files": sorted(p.name for p in OUTPUT_ROOT.iterdir()),
    "next_notebook": "03b_train_official_mini_vldit_offline_rtx6000.ipynb"
}
(OUTPUT_ROOT / "encode_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))
for p in sorted(OUTPUT_ROOT.iterdir()):
    print("-", p.name, round(p.stat().st_size / (1024 ** 2), 3), "MB")


{
  "status": "completed",
  "output_root": "/kaggle/working/gr00t_vlm_encoded_H16_batch04_full_rtx6000_offline",
  "action_horizon": 16,
  "state_horizon": 1,
  "num_prepared_roots": 1,
  "prepared_roots": [
    "/kaggle/input/notebooks/kimthanh211005/gr00t-prepare-h16-batch04-5-subsets/gr00t_prepared_official_H16_batch04"
  ],
  "root_digest": "f4ae196e89",
  "encoder_name": "/kaggle/input/notebooks/kimthanh211005/gr00t-eagle2-2b-offline-cache/eagle2_offline_cache/hf_models/nvidia_Eagle2-2B",
  "fallback_used": false,
  "primary_error": null,
  "train": {
    "split": "train",
    "cache_version": "vl_encode_multi_root_video_memmap_v1",
    "root_digest": "f4ae196e89",
    "prepared_roots": [
      "/kaggle/input/notebooks/kimthanh211005/gr00t-prepare-h16-batch04-5-subsets/gr00t_prepared_official_H16_batch04"
    ],
    "requested": 654143,
    "encoded": 654143,
    "failed": 0,
    "feature_shape": [
      654143,
      2,
      1536
    ],
    "feature_dtype": "float16",
    "enco